# Reflection and Human-in-the-Loop

**What you will build:** two patterns that make workflows more reliable — **reflection** (the model checks and improves its own output in a loop) and **human-in-the-loop** (the workflow pauses for a person to approve an action). These map to chapters 05 and 04d of the n8n course.

> **Run it:** open in [Google Colab](https://colab.research.google.com/github/ezponda/ai-agents-course/blob/main/courses/python_code/book/05_reflection_and_hitl.ipynb) (nothing to install), or run locally in Jupyter. Each notebook installs what it needs in its first cell.

In [ ]:
# Setup — same as notebook 0.1. Run once.
!pip install -q openai

import os, getpass, json
from openai import OpenAI

if not os.environ.get("OPENROUTER_API_KEY"):
    os.environ["OPENROUTER_API_KEY"] = getpass.getpass("Paste your OpenRouter API key: ")

MODEL = "meta-llama/llama-3.3-70b-instruct"
client = OpenAI(base_url="https://openrouter.ai/api/v1", api_key=os.environ["OPENROUTER_API_KEY"])

def ask(system, user, temperature=0):
    """One-shot helper: system + user in, text out."""
    r = client.chat.completions.create(
        model=MODEL, temperature=temperature,
        messages=[{"role":"system","content":system},{"role":"user","content":user}])
    return r.choices[0].message.content

print("Ready. Model:", MODEL)

## Reflection: generate, critique, improve

A single generation is one shot in the dark. Reflection adds a loop: produce a draft, check it against the requirements, and revise until it passes (or you hit a limit).

```
  generate ─▶ check ─▶ pass?  ── yes ─▶ done
     ▲                   │
     └──── revise ◀──── no
```

The most important design choice is **who does the checking**. A hierarchy, cheapest and most reliable first:

| Check type | Example | Use when |
|------------|---------|----------|
| **Code rule** | `len(words) <= 40` | The requirement is objective — always prefer this |
| **LLM judge** | "Is this friendly and on-topic?" | The requirement is subjective |

Prefer a plain Python check whenever the rule is objective — it's free, instant, and never hallucinates. Use an LLM judge only for things code can't measure.

## A reflection loop with a hard, code-checked constraint

Task: write a tagline that is **exactly 6 words** and **contains the word "quiet"**. Both are objective, so the validator is Python, not an LLM — and an exact count plus a required word is hard enough that the model reliably misses on the first try, so you actually see the loop *loop*.

In [ ]:
def word_count(text):
    return len(text.strip().split())

def write_tagline(product, max_tries=5):
    # Two objective constraints: EXACTLY 6 words AND contains 'quiet'.
    def ok(t): return word_count(t) == 6 and "quiet" in t.lower()
    draft = ask("You write punchy product taglines.",
                f"Write a tagline for {product}. It must be EXACTLY 6 words and include the word 'quiet'.")
    for attempt in range(max_tries):
        n, has = word_count(draft), ("quiet" in draft.lower())
        print(f"attempt {attempt+1}: ({n} words, has 'quiet': {has}) {draft!r}")
        if ok(draft):
            return draft                                   # passed BOTH code checks
        draft = ask("You revise taglines to fit exact constraints without losing punch.",
                    f"Product: {product}\nRewrite as EXACTLY 6 words that include the word 'quiet'.\n"
                    f"Current ({n} words, has 'quiet': {has}): {draft}")
    return draft

print("\nFINAL:", write_tagline("a noise-cancelling coffee mug that keeps drinks hot"))

## When the check is subjective: an LLM judge

Some requirements can't be measured with code ("is this reply empathetic?"). Then a second LLM call acts as a critic. Keep the critic's job narrow and make it output a machine-readable verdict so your loop can branch on it.

In [ ]:
from pydantic import BaseModel

class Verdict(BaseModel):
    passes: bool
    feedback: str

def judge(requirement, text):
    raw = client.chat.completions.create(
        model=MODEL, temperature=0, response_format={"type":"json_object"},
        messages=[{"role":"system","content":
                   f"You are a strict reviewer. Requirement: {requirement}. "
                   f"Reply JSON: {{\"passes\": bool, \"feedback\": str}}."},
                  {"role":"user","content":text}])
    return Verdict.model_validate_json(raw.choices[0].message.content)

draft = ask("You write customer support replies.", "Reply to: 'Your app deleted my data and I am furious.'")
v = judge("The reply must apologize, stay calm, and offer a concrete next step.", draft)
print("draft:", draft, "\n\nverdict:", v)

You'd wrap that `judge` call in the same generate/check/revise loop as the tagline — only the check changed. Reflection is one loop; the interesting decision is always *what the check is*.

## Human-in-the-loop: pause before acting

Some actions shouldn't happen without a person's OK — sending an email, spending money, deleting data. The pattern: the workflow *proposes*, then waits for approval before the irreversible step. In a notebook we use `input()`; in production it's a webhook or a UI (that's exactly what LangGraph's `interrupt` does in 2.4).

In [ ]:
def draft_and_send_email(request):
    draft = ask("You draft short, professional emails. Output only the email body.", request)
    print("----- PROPOSED EMAIL -----")
    print(draft)
    print("--------------------------")

    decision = input("Send this email? (y = send / n = cancel): ").strip().lower()
    if decision == "y":
        # the real 'send' would go here (SMTP, an API call, ...)
        return "Email sent."
    return "Cancelled — nothing was sent."

print(draft_and_send_email("Ask the vendor to reschedule Tuesday's demo to Thursday."))

```{tip}
The approval gate goes **around the irreversible action**, not around the whole workflow. Let the model draft freely; only stop it at the step that touches the outside world. That keeps the human in control of what matters and out of the way of what doesn't.
```

::::{dropdown} 🛠️ Try it yourself
:color: secondary

1. **Combine both checks.** In `write_tagline`, after the word-count passes, add a `judge(...)` call for "is it catchy?" and loop again if it isn't.
2. **Add an edit option.** Give the email approval a third choice, `e` = edit, that asks the user for feedback and regenerates the draft.
3. **Cap the judge loop.** Wrap the `judge` example in a full generate/check/revise loop with `max_tries=3` and print each attempt, like the tagline loop.
::::

**What's next:** in **0.6** we finish Block 0 with **memory** — how to carry a conversation across turns without blowing past the context window.